# E2-A0-R1-V2 Frozen Unseen QC Holdout -- Paper-Frozen Notebook

This notebook is a clean, minimal, self-contained re-assembly of the frozen
"E2-A0-R1-V2 -- FROZEN UNSEEN QC HOLDOUT" experiment for conference-paper use. It is meant to
run from a completely fresh Colab runtime with the Drive frozen artifacts mounted.

Every function body and every documented threshold/path/constant it uses is extracted
VERBATIM (byte-for-byte) from the archived notebook at `colab/archive/DOAR_21_8_26_V1.ipynb`
(cell indices 22, 23, and 24). See each cell's provenance banner for the exact source cell
index and item extracted. See `docs/E2_FROZEN_HOLDOUT_RUN_GUIDE.md` for exact run
instructions.

This notebook contains NO model training, NO E2-A1 experiment, NO evaluation against Locked
Test, NO parameter tuning / threshold search, and NO segmentation method beyond the frozen A0
and V2 functions extracted from the archive.


In [ ]:
# ==================================================================================================
# DOAR MASTER THESIS -- E2-A0 / R1-V2 FROZEN UNSEEN HOLDOUT -- PAPER-FROZEN NOTEBOOK
#
# This notebook is a clean, minimal, self-contained re-assembly of the frozen
# "E2-A0-R1-V2 -- FROZEN UNSEEN QC HOLDOUT" experiment (cell index 24 of the archived
# notebook colab/archive/DOAR_21_8_26_V1.ipynb), together with the frozen V2 foreground
# extractor it depends on (cell index 23 of the same archive).
#
# It contains NO model training, NO E2-A1 experiment, NO evaluation against Locked Test,
# NO parameter tuning / threshold search, and NO segmentation method beyond the frozen
# A0 / V2 functions extracted verbatim from the archive.
#
# CELL 1 -- Mount Drive + imports.
#
# Import list below is the UNION of the import blocks actually used, verbatim, by:
#   - archive cell 23 (E2-A0-R1-V2-PILOT)      -- defines build_a0_mask / build_v2_mask / QC
#   - archive cell 24 (E2-A0-R1-V2 FROZEN HOLDOUT) -- the frozen holdout execution cell
# cv2 is included because it is used internally by build_a0_mask / build_v2_mask; the
# archived holdout cell (24) itself omits "import cv2" because in the original Colab
# runtime it relied on cell 23 having already been executed in the same kernel session.
# This notebook is self-contained, so cv2 is imported explicitly here instead.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display, Image as IPImage

import hashlib
import json
import math

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm


In [ ]:
# ==================================================================================================
# CELL 2 -- REPOSITORY / VERSION PROVENANCE
#
# HONESTY NOTE:
# The archived notebook's E2 cells (indices 20-24, which is where E2-A0 / R1-V1 / R1-V2 /
# the frozen holdout live) do NOT print or pin numpy / opencv-python / pandas / matplotlib
# library versions anywhere. That is a genuine gap in the archive, not something invented
# here. (Library-version prints such as `torch.__version__` DO appear elsewhere in the
# archive, but only in the unrelated E1 representation-learning cells, which use PyTorch --
# E2 does not.) This cell records what IS knowable: the git commit of this repo at run
# time (if resolvable), this notebook's own filename/date, and the versions of the
# libraries actually installed in the runtime executing this cell -- clearly labelled as
# "observed at run time", not as a value the archive itself froze.
# ==================================================================================================

import subprocess
import sys

NOTEBOOK_SOURCE_FILENAME = "E2_A0_R1_V2_FROZEN_UNSEEN_HOLDOUT.ipynb"
ARCHIVE_SOURCE_FILENAME = "DOAR_21_8_26_V1.ipynb"
ARCHIVE_SOURCE_DATE = "2026-08-21"  # from the archive filename DOAR_21_8_26_V1 (D_M_YY)
NOTEBOOK_ASSEMBLED_DATE = "2026-08-25"

print("=" * 100)
print("REPOSITORY / VERSION PROVENANCE")
print("=" * 100)

print("Paper notebook file   :", NOTEBOOK_SOURCE_FILENAME)
print("Assembled on          :", NOTEBOOK_ASSEMBLED_DATE)
print("Frozen archive source  :", ARCHIVE_SOURCE_FILENAME)
print("Archive date (from name):", ARCHIVE_SOURCE_DATE)

try:
    commit = (
        subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            stderr=subprocess.STDOUT,
        )
        .decode("utf-8")
        .strip()
    )
    print("DOAR repo git commit   :", commit)
except Exception as exc:
    commit = None
    print("DOAR repo git commit   : UNRESOLVABLE at runtime (", exc, ")")

print()
print("Libraries observed at runtime (NOT pinned by the archive -- see honesty note above):")
print("  python  :", sys.version.split()[0])
print("  numpy   :", np.__version__)
print("  pandas  :", pd.__version__)
print("  opencv  :", cv2.__version__)
try:
    import matplotlib
    print("  matplotlib:", matplotlib.__version__)
except Exception:
    pass
try:
    import PIL
    print("  Pillow  :", PIL.__version__)
except Exception:
    pass


In [ ]:
# ==================================================================================================
# CELL 3 -- FROZEN A0 / V2 / HELPER DEFINITIONS -- EXTRACTED VERBATIM, UNCHANGED
#
# Every function body below is byte-for-byte identical to its source in
# colab/archive/DOAR_21_8_26_V1.ipynb (verified via ast.get_source_segment + substring
# check against the archived cell's raw source text -- see Step 5 static validation).
# Nothing here was retyped, paraphrased, renamed, or re-thresholded.
# ==================================================================================================
# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 23  (E2-A0-R1-V2-PILOT)
#   item: function robust_mad
# ================================================================================================
def robust_mad(values):

    values = np.asarray(
        values,
        dtype=np.float64,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:

        return 0.0

    median = np.median(
        values
    )

    mad = np.median(
        np.abs(
            values
            -
            median
        )
    )

    return float(
        1.4826
        * mad
    )

# ================================================================================================
# END VERBATIM EXTRACT (function robust_mad)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 23  (E2-A0-R1-V2-PILOT)
#   item: function get_border_pixels
# ================================================================================================
def get_border_pixels(
    image_rgb
):

    h, w, _ = (
        image_rgb.shape
    )

    by = max(
        2,
        int(
            round(
                h
                * 0.04
            )
        ),
    )

    bx = max(
        2,
        int(
            round(
                w
                * 0.04
            )
        ),
    )

    top = (
        image_rgb[
            :by,
            :,
            :
        ]
        .reshape(
            -1,
            3
        )
    )

    bottom = (
        image_rgb[
            h-by:,
            :,
            :
        ]
        .reshape(
            -1,
            3
        )
    )

    left = (
        image_rgb[
            :,
            :bx,
            :
        ]
        .reshape(
            -1,
            3
        )
    )

    right = (
        image_rgb[
            :,
            w-bx:,
            :
        ]
        .reshape(
            -1,
            3
        )
    )

    return np.concatenate(
        [
            top,
            bottom,
            left,
            right,
        ],
        axis=0,
    )

# ================================================================================================
# END VERBATIM EXTRACT (function get_border_pixels)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 23  (E2-A0-R1-V2-PILOT)
#   item: function cleanup_components
# ================================================================================================
def cleanup_components(
    raw_mask,
    minimum_fraction=0.000005,
):

    raw_mask = (
        raw_mask
        > 0
    ).astype(
        np.uint8
    )

    h, w = (
        raw_mask.shape
    )

    image_area = (
        h
        * w
    )

    min_area = max(
        3,
        int(
            round(
                image_area
                * minimum_fraction
            )
        ),
    )

    n_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            raw_mask,
            connectivity=8,
        )
    )

    cleaned = np.zeros_like(
        raw_mask
    )

    for label in range(
        1,
        n_labels
    ):

        area = stats[
            label,
            cv2.CC_STAT_AREA
        ]

        if area >= min_area:

            cleaned[
                labels
                == label
            ] = 1

    return (
        cleaned,
        min_area,
    )

# ================================================================================================
# END VERBATIM EXTRACT (function cleanup_components)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 23  (E2-A0-R1-V2-PILOT)
#   item: function build_a0_mask
# ================================================================================================
def build_a0_mask(
    image_rgb
):

    lab = cv2.cvtColor(
        image_rgb,
        cv2.COLOR_RGB2LAB,
    ).astype(
        np.float32
    )

    border_rgb = get_border_pixels(
        image_rgb
    )

    border_lab = cv2.cvtColor(
        border_rgb.reshape(
            -1,
            1,
            3
        ),
        cv2.COLOR_RGB2LAB,
    ).reshape(
        -1,
        3
    ).astype(
        np.float32
    )

    bg_lab = np.median(
        border_lab,
        axis=0,
    )

    border_dist = np.linalg.norm(
        border_lab
        -
        bg_lab[
            None,
            :
        ],
        axis=1,
    )

    background_noise = float(
        np.percentile(
            border_dist,
            98
        )
    )

    threshold = max(
        10.0,
        background_noise
        + 3.0,
    )

    distance = np.linalg.norm(
        lab
        -
        bg_lab[
            None,
            None,
            :
        ],
        axis=2,
    )

    raw_mask = (
        distance
        >
        threshold
    ).astype(
        np.uint8
    )

    cleaned, min_area = (
        cleanup_components(
            raw_mask,
            0.000005,
        )
    )

    return {
        "mask":
            cleaned,

        "threshold":
            float(
                threshold
            ),

        "background_noise":
            background_noise,

        "background_lab":
            bg_lab,

        "min_component_area":
            min_area,
    }

# ================================================================================================
# END VERBATIM EXTRACT (function build_a0_mask)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 23  (E2-A0-R1-V2-PILOT)
#   item: function build_border_selector
# ================================================================================================
def build_border_selector(
    height,
    width,
    fraction=0.06,
):

    by = max(
        2,
        int(
            round(
                height
                * fraction
            )
        ),
    )

    bx = max(
        2,
        int(
            round(
                width
                * fraction
            )
        ),
    )

    selector = np.zeros(
        (
            height,
            width
        ),
        dtype=bool,
    )

    selector[
        :by,
        :
    ] = True

    selector[
        height-by:,
        :
    ] = True

    selector[
        :,
        :bx
    ] = True

    selector[
        :,
        width-bx:
    ] = True

    return selector

# ================================================================================================
# END VERBATIM EXTRACT (function build_border_selector)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 23  (E2-A0-R1-V2-PILOT)
#   item: function border_samples_with_side_ids
# ================================================================================================
def border_samples_with_side_ids(
    image_rgb,
    fraction=0.08,
):

    h, w, _ = (
        image_rgb.shape
    )

    by = max(
        2,
        int(
            round(
                h
                * fraction
            )
        ),
    )

    bx = max(
        2,
        int(
            round(
                w
                * fraction
            )
        ),
    )

    samples = []
    side_ids = []

    top = (
        image_rgb[
            :by,
            :,
            :
        ]
        .reshape(
            -1,
            3
        )
    )

    samples.append(
        top
    )

    side_ids.extend(
        [0]
        * len(
            top
        )
    )

    bottom = (
        image_rgb[
            h-by:,
            :,
            :
        ]
        .reshape(
            -1,
            3
        )
    )

    samples.append(
        bottom
    )

    side_ids.extend(
        [1]
        * len(
            bottom
        )
    )

    left = (
        image_rgb[
            :,
            :bx,
            :
        ]
        .reshape(
            -1,
            3
        )
    )

    samples.append(
        left
    )

    side_ids.extend(
        [2]
        * len(
            left
        )
    )

    right = (
        image_rgb[
            :,
            w-bx:,
            :
        ]
        .reshape(
            -1,
            3
        )
    )

    samples.append(
        right
    )

    side_ids.extend(
        [3]
        * len(
            right
        )
    )

    rgb = np.concatenate(
        samples,
        axis=0,
    )

    side_ids = np.asarray(
        side_ids,
        dtype=np.int32,
    )

    return (
        rgb,
        side_ids,
    )

# ================================================================================================
# END VERBATIM EXTRACT (function border_samples_with_side_ids)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 23  (E2-A0-R1-V2-PILOT)
#   item: function estimate_support_v2
# ================================================================================================
def estimate_support_v2(
    image_rgb
):

    border_rgb, side_ids = (
        border_samples_with_side_ids(
            image_rgb,
            fraction=0.08,
        )
    )

    border_lab = cv2.cvtColor(
        border_rgb.reshape(
            -1,
            1,
            3
        ),
        cv2.COLOR_RGB2LAB,
    ).reshape(
        -1,
        3
    ).astype(
        np.float32
    )


    # ----------------------------------------------------------------------------------------------
    # Limit clustering size deterministically for speed.
    # ----------------------------------------------------------------------------------------------

    max_samples = 12000

    if len(
        border_lab
    ) > max_samples:

        indices = np.linspace(
            0,
            len(
                border_lab
            )
            - 1,
            max_samples,
            dtype=int,
        )

        cluster_lab = (
            border_lab[
                indices
            ]
        )

        cluster_sides = (
            side_ids[
                indices
            ]
        )

    else:

        cluster_lab = (
            border_lab
        )

        cluster_sides = (
            side_ids
        )


    # ----------------------------------------------------------------------------------------------
    # Deterministic K-means.
    # ----------------------------------------------------------------------------------------------

    cv2.setRNGSeed(
        20260824
    )

    criteria = (
        cv2.TERM_CRITERIA_EPS
        +
        cv2.TERM_CRITERIA_MAX_ITER,
        40,
        0.1,
    )

    compactness, labels, centers = (
        cv2.kmeans(
            cluster_lab,
            3,
            None,
            criteria,
            1,
            cv2.KMEANS_PP_CENTERS,
        )
    )

    labels = (
        labels
        .reshape(
            -1
        )
    )

    cluster_records = []


    for k in range(
        3
    ):

        selector = (
            labels
            == k
        )

        count = int(
            selector.sum()
        )

        if count == 0:

            continue

        ratio = (
            count
            /
            len(
                labels
            )
        )

        values = (
            cluster_lab[
                selector
            ]
        )

        center = (
            centers[
                k
            ]
        )

        distances = np.linalg.norm(
            values
            -
            center[
                None,
                :
            ],
            axis=1,
        )

        dispersion = float(
            np.median(
                distances
            )
        )

        dispersion_scale = (
            robust_mad(
                distances
            )
        )

        covered_sides = 0

        side_fractions = []

        for side in range(
            4
        ):

            side_selector = (
                cluster_sides
                == side
            )

            side_total = int(
                side_selector.sum()
            )

            if side_total == 0:

                side_fraction = 0.0

            else:

                side_fraction = float(
                    (
                        selector
                        &
                        side_selector
                    ).sum()
                    /
                    side_total
                )

            side_fractions.append(
                side_fraction
            )

            if (
                side_fraction
                >= 0.05
            ):

                covered_sides += 1


        side_factor = (
            0.70
            +
            0.30
            * (
                covered_sides
                /
                4.0
            )
        )

        compactness_factor = (
            1.0
            /
            (
                1.0
                +
                dispersion
                /
                25.0
            )
        )

        score = (
            ratio
            *
            side_factor
            *
            compactness_factor
        )


        cluster_records.append(
            {
                "cluster":
                    k,

                "count":
                    count,

                "ratio":
                    ratio,

                "covered_sides":
                    covered_sides,

                "side_fractions":
                    side_fractions,

                "dispersion":
                    dispersion,

                "dispersion_scale":
                    dispersion_scale,

                "score":
                    score,

                "center":
                    center,
            }
        )


    best = max(
        cluster_records,
        key=lambda x: x[
            "score"
        ],
    )


    support_center = (
        np.asarray(
            best[
                "center"
            ],
            dtype=np.float32,
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Support radius around dominant cluster.
    # ----------------------------------------------------------------------------------------------

    best_selector = (
        labels
        == best[
            "cluster"
        ]
    )

    best_values = (
        cluster_lab[
            best_selector
        ]
    )

    best_distances = np.linalg.norm(
        best_values
        -
        support_center[
            None,
            :
        ],
        axis=1,
    )

    median_distance = float(
        np.median(
            best_distances
        )
    )

    distance_scale = (
        robust_mad(
            best_distances
        )
    )

    support_radius = (
        median_distance
        +
        4.0
        * distance_scale
        +
        2.0
    )

    support_radius = float(
        np.clip(
            support_radius,
            8.0,
            28.0,
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Determine how much of whole image resembles that support.
    # ----------------------------------------------------------------------------------------------

    h, w = (
        image_rgb.shape[
            :2
        ]
    )

    scale = min(
        1.0,
        320.0
        /
        max(
            h,
            w
        ),
    )

    small_w = max(
        32,
        int(
            round(
                w
                * scale
            )
        ),
    )

    small_h = max(
        32,
        int(
            round(
                h
                * scale
            )
        ),
    )

    small_rgb = cv2.resize(
        image_rgb,
        (
            small_w,
            small_h
        ),
        interpolation=cv2.INTER_AREA,
    )

    small_lab = cv2.cvtColor(
        small_rgb,
        cv2.COLOR_RGB2LAB,
    ).astype(
        np.float32
    )

    whole_distance = np.linalg.norm(
        small_lab
        -
        support_center[
            None,
            None,
            :
        ],
        axis=2,
    )

    whole_support_fraction = float(
        (
            whole_distance
            <= support_radius
        ).mean()
    )


    dominant_ratio = float(
        best[
            "ratio"
        ]
    )

    covered_sides = int(
        best[
            "covered_sides"
        ]
    )

    dispersion = float(
        best[
            "dispersion"
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # Support confidence.
    # ----------------------------------------------------------------------------------------------

    if (
        dominant_ratio
        >= 0.55
        and
        covered_sides
        == 4
        and
        whole_support_fraction
        >= 0.20
        and
        dispersion
        <= 18.0
    ):

        confidence = (
            "HIGH"
        )

    elif (
        dominant_ratio
        >= 0.35
        and
        covered_sides
        >= 3
        and
        whole_support_fraction
        >= 0.12
        and
        dispersion
        <= 25.0
    ):

        confidence = (
            "MEDIUM"
        )

    else:

        confidence = (
            "LOW"
        )


    # ----------------------------------------------------------------------------------------------
    # Support regime.
    #
    # OpenCV LAB L is 0–255.
    # ----------------------------------------------------------------------------------------------

    L = float(
        support_center[
            0
        ]
    )

    if confidence == "LOW":

        regime = (
            "AMBIGUOUS_FULL_CANVAS"
        )

    elif L >= 170:

        regime = (
            "LIGHT_SUPPORT"
        )

    elif L <= 90:

        regime = (
            "DARK_SUPPORT"
        )

    else:

        regime = (
            "MID_SUPPORT"
        )


    return {
        "support_center_lab":
            support_center,

        "support_radius":
            support_radius,

        "dominant_border_ratio":
            dominant_ratio,

        "covered_sides":
            covered_sides,

        "whole_support_fraction":
            whole_support_fraction,

        "cluster_dispersion":
            dispersion,

        "support_confidence":
            confidence,

        "support_regime":
            regime,

        "cluster_score":
            float(
                best[
                    "score"
                ]
            ),
    }

# ================================================================================================
# END VERBATIM EXTRACT (function estimate_support_v2)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 23  (E2-A0-R1-V2-PILOT)
#   item: function suppress_smooth_border_floods
# ================================================================================================
def suppress_smooth_border_floods(
    mask,
    local_residual,
    saturation_difference,
    local_threshold,
):

    mask = (
        mask
        > 0
    ).astype(
        np.uint8
    )

    h, w = (
        mask.shape
    )

    image_area = (
        h
        * w
    )

    n_labels, labels, stats, _ = (
        cv2.connectedComponentsWithStats(
            mask,
            connectivity=8,
        )
    )

    result = (
        mask.copy()
    )


    for label in range(
        1,
        n_labels
    ):

        component = (
            labels
            == label
        )

        area = int(
            stats[
                label,
                cv2.CC_STAT_AREA
            ]
        )

        area_fraction = (
            area
            /
            image_area
        )

        x = int(
            stats[
                label,
                cv2.CC_STAT_LEFT
            ]
        )

        y = int(
            stats[
                label,
                cv2.CC_STAT_TOP
            ]
        )

        width = int(
            stats[
                label,
                cv2.CC_STAT_WIDTH
            ]
        )

        height = int(
            stats[
                label,
                cv2.CC_STAT_HEIGHT
            ]
        )

        touches_border = (
            x <= 1
            or
            y <= 1
            or
            (
                x
                +
                width
            )
            >= (
                w
                - 1
            )
            or
            (
                y
                +
                height
            )
            >= (
                h
                - 1
            )
        )


        if (
            not touches_border
            or
            area_fraction
            < 0.15
        ):

            continue


        mean_local = float(
            np.mean(
                local_residual[
                    component
                ]
            )
        )

        mean_sat_difference = float(
            np.mean(
                saturation_difference[
                    component
                ]
            )
        )


        # ------------------------------------------------------------------------------------------
        # Large smooth border component:
        # likely page shadow / lighting / external border rather than graphic content.
        # ------------------------------------------------------------------------------------------

        if (
            mean_local
            <
            (
                local_threshold
                * 1.15
            )
            and
            mean_sat_difference
            <
            18.0
        ):

            result[
                component
            ] = 0


    return result

# ================================================================================================
# END VERBATIM EXTRACT (function suppress_smooth_border_floods)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 23  (E2-A0-R1-V2-PILOT)
#   item: function build_v2_mask
# ================================================================================================
def build_v2_mask(
    image_rgb
):

    lab = cv2.cvtColor(
        image_rgb,
        cv2.COLOR_RGB2LAB,
    ).astype(
        np.float32
    )

    hsv = cv2.cvtColor(
        image_rgb,
        cv2.COLOR_RGB2HSV,
    ).astype(
        np.float32
    )

    gray = cv2.cvtColor(
        image_rgb,
        cv2.COLOR_RGB2GRAY,
    ).astype(
        np.float32
    )


    L_channel = (
        lab[
            :,
            :,
            0
        ]
    )

    saturation = (
        hsv[
            :,
            :,
            1
        ]
    )


    support = estimate_support_v2(
        image_rgb
    )

    support_center = (
        support[
            "support_center_lab"
        ]
    )

    support_radius = float(
        support[
            "support_radius"
        ]
    )

    regime = (
        support[
            "support_regime"
        ]
    )

    confidence = (
        support[
            "support_confidence"
        ]
    )


    h, w = (
        gray.shape
    )

    image_area = (
        h
        * w
    )


    # ----------------------------------------------------------------------------------------------
    # Distance from dominant support colour.
    # ----------------------------------------------------------------------------------------------

    distance = np.linalg.norm(
        lab
        -
        support_center[
            None,
            None,
            :
        ],
        axis=2,
    )


    # ----------------------------------------------------------------------------------------------
    # Slowly varying local background.
    #
    # This reduces sensitivity to photo lighting gradients.
    # ----------------------------------------------------------------------------------------------

    sigma = max(
        5.0,
        min(
            h,
            w
        )
        /
        55.0,
    )


    local_L_background = cv2.GaussianBlur(
        L_channel,
        (
            0,
            0
        ),
        sigmaX=sigma,
        sigmaY=sigma,
    )


    local_residual = np.abs(
        L_channel
        -
        local_L_background
    )


    local_sat_background = cv2.GaussianBlur(
        saturation,
        (
            0,
            0
        ),
        sigmaX=sigma,
        sigmaY=sigma,
    )


    local_sat_residual = np.abs(
        saturation
        -
        local_sat_background
    )


    # ==============================================================================================
    # 11A. AMBIGUOUS / FULL-CANVAS MODE
    #
    # We do NOT claim to know the blank-page/background support.
    #
    # Build conservative structural-content mask only for visualization.
    # Final QC remains AMBIGUOUS_SUPPORT.
    # ==============================================================================================

    if regime == "AMBIGUOUS_FULL_CANVAS":

        local_scale = robust_mad(
            local_residual.reshape(
                -1
            )
        )

        local_threshold = float(
            np.clip(
                np.median(
                    local_residual
                )
                +
                4.5
                * local_scale,
                6.0,
                22.0,
            )
        )


        sat_scale = robust_mad(
            local_sat_residual.reshape(
                -1
            )
        )

        sat_threshold = float(
            np.clip(
                np.median(
                    local_sat_residual
                )
                +
                4.0
                * sat_scale,
                8.0,
                35.0,
            )
        )


        gray_u8 = np.clip(
            gray,
            0,
            255,
        ).astype(
            np.uint8
        )


        edges = cv2.Canny(
            gray_u8,
            50,
            150,
        ) > 0


        edge_kernel = np.ones(
            (
                2,
                2
            ),
            dtype=np.uint8,
        )


        edges = (
            cv2.dilate(
                edges.astype(
                    np.uint8
                ),
                edge_kernel,
                iterations=1,
            )
            > 0
        )


        raw = (
            (
                local_residual
                >
                local_threshold
            )
            |
            (
                local_sat_residual
                >
                sat_threshold
            )
            |
            edges
        ).astype(
            np.uint8
        )


        cleaned, min_area = (
            cleanup_components(
                raw,
                0.000003,
            )
        )


        final_mask = (
            cleaned
        )


        return {
            "mask":
                final_mask,

            "support_regime":
                regime,

            "support_confidence":
                confidence,

            "support_radius":
                support_radius,

            "dominant_border_ratio":
                support[
                    "dominant_border_ratio"
                ],

            "covered_sides":
                support[
                    "covered_sides"
                ],

            "whole_support_fraction":
                support[
                    "whole_support_fraction"
                ],

            "cluster_dispersion":
                support[
                    "cluster_dispersion"
                ],

            "global_threshold":
                np.nan,

            "local_threshold":
                local_threshold,

            "sat_threshold":
                sat_threshold,

            "mask_strategy":
                "AMBIGUOUS_LOCAL_CONTENT_ONLY",

            "min_component_area":
                min_area,
        }


    # ==============================================================================================
    # 11B. RELIABLE SUPPORT
    # ==============================================================================================

    support_selector = (
        distance
        <= support_radius
    )


    # ----------------------------------------------------------------------------------------------
    # Guard against too few support pixels.
    # ----------------------------------------------------------------------------------------------

    if (
        support_selector.sum()
        <
        max(
            100,
            int(
                0.002
                * image_area
            ),
        )
    ):

        # Conservative fallback.
        support_selector = (
            distance
            <= (
                support_radius
                * 1.5
            )
        )


    support_L = (
        L_channel[
            support_selector
        ]
    )

    support_sat = (
        saturation[
            support_selector
        ]
    )

    support_local_residual = (
        local_residual[
            support_selector
        ]
    )


    bg_L = float(
        np.median(
            support_L
        )
    )

    bg_sat = float(
        np.median(
            support_sat
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Global support-difference threshold.
    # ----------------------------------------------------------------------------------------------

    global_threshold = float(
        np.clip(
            support_radius
            +
            6.0,
            14.0,
            42.0,
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Local line/stroke threshold.
    # ----------------------------------------------------------------------------------------------

    local_threshold = (
        np.median(
            support_local_residual
        )
        +
        5.0
        * robust_mad(
            support_local_residual
        )
    )

    local_threshold = float(
        np.clip(
            local_threshold,
            5.0,
            18.0,
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Luminance deviation threshold.
    #
    # Symmetric:
    # works for dark ink on light support AND light chalk on dark support.
    # ----------------------------------------------------------------------------------------------

    L_scale = robust_mad(
        support_L
    )

    luminance_delta = float(
        np.clip(
            max(
                22.0,
                4.0
                * L_scale,
            ),
            22.0,
            55.0,
        )
    )


    luminance_difference = np.abs(
        L_channel
        -
        bg_L
    )


    strong_luminance = (
        luminance_difference
        >
        luminance_delta
    )


    # ----------------------------------------------------------------------------------------------
    # Chromatic difference.
    # ----------------------------------------------------------------------------------------------

    sat_scale = robust_mad(
        support_sat
    )

    saturation_delta = float(
        np.clip(
            max(
                22.0,
                4.0
                * sat_scale,
            ),
            22.0,
            60.0,
        )
    )


    saturation_difference = np.abs(
        saturation
        -
        bg_sat
    )


    chromatic_difference = (
        saturation_difference
        >
        saturation_delta
    )


    # ----------------------------------------------------------------------------------------------
    # Local structure.
    # ----------------------------------------------------------------------------------------------

    local_structure = (
        local_residual
        >
        local_threshold
    )


    # ----------------------------------------------------------------------------------------------
    # Candidate foreground.
    #
    # A pixel can become foreground if:
    #
    # A) local line/edge evidence
    #
    # OR
    #
    # B) it is globally different from support AND has strong luminance
    #    or chromatic evidence.
    #
    # This prevents broad mild illumination gradients from entering solely
    # because they differ somewhat from one global support centre.
    # ----------------------------------------------------------------------------------------------

    global_difference = (
        distance
        >
        global_threshold
    )


    raw = (
        local_structure
        |
        (
            global_difference
            &
            (
                strong_luminance
                |
                chromatic_difference
            )
        )
    ).astype(
        np.uint8
    )


    # ----------------------------------------------------------------------------------------------
    # Remove huge smooth border-connected floods.
    # ----------------------------------------------------------------------------------------------

    raw = suppress_smooth_border_floods(
        mask=
            raw,

        local_residual=
            local_residual,

        saturation_difference=
            saturation_difference,

        local_threshold=
            local_threshold,
    )


    cleaned, min_area = (
        cleanup_components(
            raw,
            0.000003,
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Minimal closing only.
    # ----------------------------------------------------------------------------------------------

    kernel = np.ones(
        (
            2,
            2
        ),
        dtype=np.uint8,
    )


    cleaned = cv2.morphologyEx(
        cleaned,
        cv2.MORPH_CLOSE,
        kernel,
        iterations=1,
    )


    return {
        "mask":
            cleaned,

        "support_regime":
            regime,

        "support_confidence":
            confidence,

        "support_radius":
            support_radius,

        "dominant_border_ratio":
            support[
                "dominant_border_ratio"
            ],

        "covered_sides":
            support[
                "covered_sides"
            ],

        "whole_support_fraction":
            support[
                "whole_support_fraction"
            ],

        "cluster_dispersion":
            support[
                "cluster_dispersion"
            ],

        "global_threshold":
            global_threshold,

        "local_threshold":
            local_threshold,

        "sat_threshold":
            saturation_delta,

        "mask_strategy":
            "SUPPORT_AWARE_GLOBAL_PLUS_LOCAL",

        "min_component_area":
            min_area,
    }

# ================================================================================================
# END VERBATIM EXTRACT (function build_v2_mask)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 23  (E2-A0-R1-V2-PILOT)
#   item: function border_foreground_fraction
# ================================================================================================
def border_foreground_fraction(
    mask,
    fraction=0.05,
):

    h, w = (
        mask.shape
    )

    selector = (
        build_border_selector(
            h,
            w,
            fraction,
        )
    )

    return float(
        (
            mask[
                selector
            ]
            > 0
        ).mean()
    )

# ================================================================================================
# END VERBATIM EXTRACT (function border_foreground_fraction)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 23  (E2-A0-R1-V2-PILOT)
#   item: function v2_qc_status
# ================================================================================================
def v2_qc_status(
    foreground_ratio,
    border_fg_ratio,
    support_regime,
):

    if (
        support_regime
        == "AMBIGUOUS_FULL_CANVAS"
    ):

        return (
            "AMBIGUOUS_SUPPORT"
        )

    if foreground_ratio <= 0:

        return (
            "NO_FOREGROUND"
        )

    if foreground_ratio < 0.001:

        return (
            "VERY_LOW_FOREGROUND"
        )

    if (
        foreground_ratio
        > 0.80
    ):

        return (
            "VERY_HIGH_FOREGROUND"
        )

    if (
        foreground_ratio
        > 0.50
        and
        border_fg_ratio
        > 0.75
    ):

        return (
            "POSSIBLE_BACKGROUND_FLOOD"
        )

    return (
        "OK"
    )

# ================================================================================================
# END VERBATIM EXTRACT (function v2_qc_status)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: function sha256_file
# ================================================================================================
def sha256_file(path):

    hasher = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            hasher.update(chunk)

    return hasher.hexdigest()

# ================================================================================================
# END VERBATIM EXTRACT (function sha256_file)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: function deterministic_hash
# ================================================================================================
def deterministic_hash(
    image_id,
    salt,
):

    return hashlib.sha256(
        (
            str(salt)
            +
            "::"
            +
            str(image_id)
        ).encode(
            "utf-8"
        )
    ).hexdigest()

# ================================================================================================
# END VERBATIM EXTRACT (function deterministic_hash)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: function proportional_stratified_sample
# ================================================================================================
def proportional_stratified_sample(
    dataframe,
    n_total,
    salt,
):

    df = (
        dataframe.copy()
    )


    counts = (
        df.groupby(
            STRATA,
            observed=True,
        )
        .size()
        .reset_index(
            name="available"
        )
    )


    active_count = len(
        counts
    )


    if n_total < active_count:

        raise ValueError(
            "Requested sample is smaller than number of non-empty strata."
        )


    # ----------------------------------------------------------------------------------------------
    # Give each stratum one sample first.
    # ----------------------------------------------------------------------------------------------

    counts[
        "quota"
    ] = 1


    remaining = (
        n_total
        -
        int(
            counts[
                "quota"
            ].sum()
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Allocate remaining quota iteratively in proportion to available capacity.
    # ----------------------------------------------------------------------------------------------

    while remaining > 0:

        counts[
            "capacity"
        ] = (
            counts[
                "available"
            ]
            -
            counts[
                "quota"
            ]
        )


        eligible = (
            counts[
                "capacity"
            ]
            > 0
        )


        if not eligible.any():

            raise RuntimeError(
                "Not enough samples available to satisfy requested total."
            )


        total_capacity = float(
            counts.loc[
                eligible,
                "capacity"
            ].sum()
        )


        counts[
            "_allocation_score"
        ] = -1.0


        counts.loc[
            eligible,
            "_allocation_score"
        ] = (
            counts.loc[
                eligible,
                "capacity"
            ]
            /
            total_capacity
        )


        # Deterministically allocate one at a time to highest-capacity proportion.
        best_index = (
            counts[
                "_allocation_score"
            ]
            .idxmax()
        )


        counts.loc[
            best_index,
            "quota"
        ] += 1


        remaining -= 1


    sampled_parts = []


    for _, quota_row in counts.iterrows():

        subset = df.copy()


        for column in STRATA:

            subset = subset[
                subset[
                    column
                ]
                ==
                quota_row[
                    column
                ]
            ]


        subset = (
            subset.copy()
        )


        subset[
            "_selection_hash"
        ] = subset[
            "image_id"
        ].map(
            lambda x:
            deterministic_hash(
                x,
                salt,
            )
        )


        subset = (
            subset
            .sort_values(
                "_selection_hash"
            )
            .head(
                int(
                    quota_row[
                        "quota"
                    ]
                )
            )
        )


        sampled_parts.append(
            subset
        )


    sampled = pd.concat(
        sampled_parts,
        ignore_index=True,
    )


    sampled = sampled.drop(
        columns=[
            "_selection_hash"
        ],
        errors="ignore",
    )


    assert len(sampled) == n_total
    assert sampled["image_id"].is_unique


    return (
        sampled,
        counts.drop(
            columns=[
                "_allocation_score",
                "capacity",
            ],
            errors="ignore",
        ),
    )

# ================================================================================================
# END VERBATIM EXTRACT (function proportional_stratified_sample)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: function measurement_policy
# ================================================================================================
def measurement_policy(
    qc_status,
    support_confidence,
):

    if qc_status in [
        "AMBIGUOUS_SUPPORT",
        "NO_FOREGROUND",
        "VERY_LOW_FOREGROUND",
    ]:

        return (
            "MASK_UNRELIABLE"
        )


    if qc_status in [
        "POSSIBLE_BACKGROUND_FLOOD",
        "VERY_HIGH_FOREGROUND",
    ]:

        return (
            "MASK_REVIEW_REQUIRED"
        )


    if (
        support_confidence
        in [
            "HIGH",
            "MEDIUM",
        ]
        and
        qc_status
        == "OK"
    ):

        return (
            "MASK_USABLE"
        )


    return (
        "MASK_REVIEW_REQUIRED"
    )

# ================================================================================================
# END VERBATIM EXTRACT (function measurement_policy)
# ================================================================================================

# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: function create_holdout_contact_sheets
# ================================================================================================
def create_holdout_contact_sheets(
    dataframe,
    output_directory,
    base_name,
    cases_per_page=4,
):

    output_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    dataframe = (
        dataframe
        .sort_values(
            [
                "split",
                "true_class",
                "image_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    total_pages = math.ceil(
        len(
            dataframe
        )
        /
        cases_per_page
    )


    pdf_path = (
        output_directory
        /
        f"{base_name}.pdf"
    )


    png_pages = []


    with PdfPages(
        pdf_path
    ) as pdf:

        for page_idx in tqdm(
            range(
                total_pages
            ),
            desc=base_name,
        ):

            start = (
                page_idx
                *
                cases_per_page
            )

            end = min(
                start
                +
                cases_per_page,
                len(
                    dataframe
                ),
            )


            page_df = (
                dataframe
                .iloc[
                    start:end
                ]
                .reset_index(
                    drop=True
                )
            )


            fig, axes = plt.subplots(
                cases_per_page,
                3,
                figsize=(
                    14,
                    4
                    *
                    cases_per_page
                ),
            )


            axes = np.asarray(
                axes
            ).reshape(
                cases_per_page,
                3
            )


            for axis in axes.flat:

                axis.axis(
                    "off"
                )


            for r, (_, row) in enumerate(
                page_df.iterrows()
            ):

                image_path = (
                    DATASET_ROOT
                    /
                    row[
                        "relative_path"
                    ]
                )


                with Image.open(
                    image_path
                ) as image_file:

                    image_rgb = np.asarray(
                        image_file.convert(
                            "RGB"
                        )
                    )


                a0 = build_a0_mask(
                    image_rgb
                )

                v2 = build_v2_mask(
                    image_rgb
                )


                axes[
                    r,
                    0
                ].imshow(
                    image_rgb
                )

                axes[
                    r,
                    0
                ].set_title(
                    (
                        "ORIGINAL\n"
                        f"{row['true_class']} | "
                        f"{row['split']}\n"
                        f"{str(row['image_id'])[:10]}..."
                    ),
                    fontsize=8,
                )


                axes[
                    r,
                    1
                ].imshow(
                    a0[
                        "mask"
                    ],
                    cmap="gray",
                    vmin=0,
                    vmax=1,
                )

                axes[
                    r,
                    1
                ].set_title(
                    (
                        "A0\n"
                        f"FG={a0['mask'].mean():.3f}\n"
                        f"T={a0['threshold']:.1f}"
                    ),
                    fontsize=8,
                )


                axes[
                    r,
                    2
                ].imshow(
                    v2[
                        "mask"
                    ],
                    cmap="gray",
                    vmin=0,
                    vmax=1,
                )

                axes[
                    r,
                    2
                ].set_title(
                    (
                        "FROZEN V2\n"
                        f"FG={v2['mask'].mean():.3f}\n"
                        f"{v2['support_regime']}\n"
                        f"CONF={v2['support_confidence']}"
                    ),
                    fontsize=8,
                )


                for c in range(
                    3
                ):

                    axes[
                        r,
                        c
                    ].axis(
                        "off"
                    )


            fig.suptitle(
                (
                    f"{base_name}\n"
                    f"UNSEEN HOLDOUT — Page "
                    f"{page_idx + 1}/{total_pages}"
                ),
                fontsize=14,
            )


            fig.tight_layout(
                rect=[
                    0,
                    0,
                    1,
                    0.975,
                ]
            )


            png_path = (
                output_directory
                /
                (
                    f"{base_name}_"
                    f"page_{page_idx + 1:03d}.png"
                )
            )


            fig.savefig(
                png_path,
                dpi=180,
                bbox_inches="tight",
            )


            pdf.savefig(
                fig,
                bbox_inches="tight",
            )


            plt.close(
                fig
            )


            png_pages.append(
                png_path
            )


    return {
        "pdf":
            pdf_path,

        "png_pages":
            png_pages,
    }

# ================================================================================================
# END VERBATIM EXTRACT (function create_holdout_contact_sheets)
# ================================================================================================



In [ ]:
# ==================================================================================================
# CELL 4 -- FROZEN ARTIFACT / PATH VERIFICATION
#
# Path constants and the SHA256 gate below are copied verbatim from archive cell 24
# (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT). These are the exact Drive paths the frozen
# experiment reads from. Nothing here is a new/guessed path.
# ==================================================================================================
# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: required-frozen-function presence check
# ================================================================================================
required_functions = [
    "build_v2_mask",
    "build_a0_mask",
    "border_foreground_fraction",
    "v2_qc_status",
]

missing_functions = [
    name
    for name in required_functions
    if name not in globals()
]

if missing_functions:

    raise RuntimeError(
        "\n"
        "STOP — the frozen V2 functions are not loaded in this runtime.\n\n"
        f"Missing: {missing_functions}\n\n"
        "Rerun the SAME E2-A0-R1-V2 DEVELOPMENT PILOT cell that you already used.\n"
        "Do NOT edit its thresholds or implementation.\n"
        "Then rerun this holdout cell."
    )

# ================================================================================================
# END VERBATIM EXTRACT (required-frozen-function presence check)
# ================================================================================================


# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: path constants + directory creation
# ================================================================================================
DATASET_ROOT = Path(
    "/content/drive/MyDrive/Masters/Datasets/Combined_Drawing"
)

E2A0_ROOT = Path(
    "/content/drive/MyDrive/Masters/DOAR_GPU_RESULTS/E2/"
    "E2A_INTERPRETABLE_FEATURES"
)

SOURCE_FEATURE_FILE = (
    E2A0_ROOT
    / "tables"
    / "E2A_T1_INTERPRETABLE_FEATURES.csv"
)

V1_RESULT_FILE = (
    E2A0_ROOT
    / "E2A0_R1_PILOT_V1"
    / "tables"
    / "E2A0_R1_PILOT_MASK_RESULTS.csv"
)

V2_DEV_RESULT_FILE = (
    E2A0_ROOT
    / "E2A0_R1_V2_PILOT"
    / "tables"
    / "E2A0_R1_V2_PILOT_RESULTS.csv"
)

HOLDOUT_ROOT = (
    E2A0_ROOT
    / "E2A0_R1_V2_UNSEEN_HOLDOUT"
)

TABLE_DIR = (
    HOLDOUT_ROOT
    / "tables"
)

FIGURE_DIR = (
    HOLDOUT_ROOT
    / "figures"
)

FLAGGED_CONTACT_DIR = (
    FIGURE_DIR
    / "flagged_contact_sheets"
)

CONTROL_CONTACT_DIR = (
    FIGURE_DIR
    / "control_contact_sheets"
)

for directory in [
    HOLDOUT_ROOT,
    TABLE_DIR,
    FIGURE_DIR,
    FLAGGED_CONTACT_DIR,
    CONTROL_CONTACT_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ==================================================================================================
# 3. VERIFY FROZEN E2-A0 SOURCE
# ==================================================================================================

EXPECTED_E2A0_SHA256 = (
    "6ea50e3a9517e4681eda1f05e802b384e07b54295850d43fa8cd0a7dd78d1fda"
)

# ================================================================================================
# END VERBATIM EXTRACT (path constants + directory creation)
# ================================================================================================


# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: frozen E2-A0 SHA256 verification
# ================================================================================================
assert SOURCE_FEATURE_FILE.exists()
assert V1_RESULT_FILE.exists()
assert V2_DEV_RESULT_FILE.exists()


source_sha = sha256_file(
    SOURCE_FEATURE_FILE
)


assert source_sha == EXPECTED_E2A0_SHA256, (
    "STOP: frozen E2-A0 source changed."
)


print()
print("✓ E2-A0 SHA verified:")
print(source_sha)

# ================================================================================================
# END VERBATIM EXTRACT (frozen E2-A0 SHA256 verification)
# ================================================================================================


# --------------------------------------------------------------------------------------------------
# Explicit existence checks for every frozen artifact path referenced above (os.path.exists style).
# This is new glue code (not present verbatim in the archive as a single block) but every path
# and file name it checks is copied verbatim from the archive -- no substitutions.
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("FROZEN ARTIFACT / PATH EXISTENCE CHECKS")
print("=" * 100)

for label, path in [
    ("DATASET_ROOT", DATASET_ROOT),
    ("E2A0_ROOT", E2A0_ROOT),
    ("SOURCE_FEATURE_FILE", SOURCE_FEATURE_FILE),
    ("V1_RESULT_FILE (84-dev, R1-V1 pilot)", V1_RESULT_FILE),
    ("V2_DEV_RESULT_FILE (84-dev, R1-V2 pilot)", V2_DEV_RESULT_FILE),
    ("HOLDOUT_ROOT", HOLDOUT_ROOT),
]:
    status = "OK" if Path(path).exists() else "MISSING"
    print(f"  [{status:7s}] {label}: {path}")

print()
print("Note: DATASET_ROOT / E2A0_ROOT / SOURCE_FEATURE_FILE / V1_RESULT_FILE /")
print("V2_DEV_RESULT_FILE MUST all exist for this notebook to run. If any are")
print("MISSING, the frozen Drive artifacts have not been mounted correctly --")
print("STOP and fix the Drive mount before proceeding.")


In [ ]:
# ==================================================================================================
# CELL 5 -- HOLDOUT INTEGRITY CHECKS
#
# The frozen 80-image holdout is NOT stored as a separate static manifest file that this
# notebook loads -- in the archive it is REGENERATED deterministically (SHA256-salted,
# proportional-stratified sampling; see proportional_stratified_sample in CELL 3) from the
# frozen E2-A0 feature table, the frozen 84-image development set, and two fixed salt
# strings. Because the sampling is deterministic (not random) and the inputs are SHA256-
# verified as unchanged (CELL 4), re-running this exact verbatim logic reproduces the
# identical 80-image holdout every time. This cell rebuilds it and asserts every integrity
# property the task requires.
#
# HONESTY NOTE: the frozen holdout's group label for the second half of the split is
# 'A0_OK' in the archive (not 'A0_OK_CONTROL'). Both refer to the same thing -- 40
# previously-unseen images that A0 did NOT flag -- but this notebook preserves the
# archive's exact label rather than renaming it.
# ==================================================================================================
# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: load source data + identify development images + create unseen pools
# ================================================================================================
features = pd.read_csv(
    SOURCE_FEATURE_FILE
)

features["image_id"] = (
    features["image_id"]
    .astype(str)
)


assert len(features) == 2883
assert features["image_id"].is_unique

assert (
    features["split"]
    == "train"
).sum() == 2599

assert (
    features["split"]
    == "valid"
).sum() == 284

assert (
    features["split"]
    == "test"
).sum() == 0


# ==================================================================================================
# 5. IDENTIFY ALL DEVELOPMENT IMAGES THAT MUST BE EXCLUDED
# ==================================================================================================

v1_dev = pd.read_csv(
    V1_RESULT_FILE
)

v1_dev["image_id"] = (
    v1_dev["image_id"]
    .astype(str)
)


v2_dev = pd.read_csv(
    V2_DEV_RESULT_FILE
)

v2_dev["image_id"] = (
    v2_dev["image_id"]
    .astype(str)
)


development_ids = set(
    v1_dev["image_id"]
).union(
    set(
        v2_dev["image_id"]
    )
)


assert len(development_ids) == 84, (
    f"Expected 84 unique development images; found {len(development_ids)}."
)


print()
print("✓ Development images excluded:", len(development_ids))


# ==================================================================================================
# 6. CREATE UNSEEN POOLS
# ==================================================================================================

unseen = (
    features[
        ~features[
            "image_id"
        ].isin(
            development_ids
        )
    ]
    .copy()
)


unseen[
    "selection_group"
] = np.where(
    unseen[
        "qc_flag"
    ].isin(
        [
            "NO_FOREGROUND",
            "VERY_LOW_FOREGROUND",
        ]
    ),
    "A0_FLAGGED",
    "A0_OK",
)


flagged_pool = (
    unseen[
        unseen[
            "selection_group"
        ]
        == "A0_FLAGGED"
    ]
    .copy()
)


ok_pool = (
    unseen[
        unseen[
            "selection_group"
        ]
        == "A0_OK"
    ]
    .copy()
)


assert len(flagged_pool) >= 40
assert len(ok_pool) >= 40


print()
print("Unseen A0-flagged pool:", len(flagged_pool))
print("Unseen A0-OK pool     :", len(ok_pool))

# ================================================================================================
# END VERBATIM EXTRACT (load source data + identify development images + create unseen pools)
# ================================================================================================


# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: STRATA constant
# ================================================================================================
STRATA = [
    "split",
    "true_class",
]

# ================================================================================================
# END VERBATIM EXTRACT (STRATA constant)
# ================================================================================================


# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via ast.get_source_segment on the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: select the frozen 80-image holdout + save manifest/quotas
# ================================================================================================
holdout_flagged, flagged_quota = (
    proportional_stratified_sample(
        flagged_pool,
        n_total=40,
        salt="DOAR_E2A0_R1_V2_HOLDOUT_FLAGGED_V1",
    )
)


holdout_ok, ok_quota = (
    proportional_stratified_sample(
        ok_pool,
        n_total=40,
        salt="DOAR_E2A0_R1_V2_HOLDOUT_OK_V1",
    )
)


holdout_flagged[
    "holdout_group"
] = "A0_FLAGGED"


holdout_ok[
    "holdout_group"
] = "A0_OK"


holdout = pd.concat(
    [
        holdout_flagged,
        holdout_ok,
    ],
    ignore_index=True,
)


assert len(holdout) == 80
assert holdout["image_id"].is_unique

assert not set(
    holdout["image_id"]
).intersection(
    development_ids
)


HOLDOUT_MANIFEST_FILE = (
    TABLE_DIR
    / "E2A0_R1_V2_UNSEEN_HOLDOUT_MANIFEST.csv"
)


holdout.to_csv(
    HOLDOUT_MANIFEST_FILE,
    index=False,
)


flagged_quota.to_csv(
    TABLE_DIR
    / "E2A0_R1_V2_HOLDOUT_FLAGGED_QUOTAS.csv",
    index=False,
)


ok_quota.to_csv(
    TABLE_DIR
    / "E2A0_R1_V2_HOLDOUT_OK_QUOTAS.csv",
    index=False,
)


print()
print("=" * 112)
print("UNSEEN HOLDOUT CREATED")
print("=" * 112)

print("A0 flagged:", len(holdout_flagged))
print("A0 OK     :", len(holdout_ok))
print("Total     :", len(holdout))

print()
print("Split × class composition:")

print(
    pd.crosstab(
        [
            holdout[
                "holdout_group"
            ],
            holdout[
                "split"
            ],
        ],
        holdout[
            "true_class"
        ],
        margins=True,
    )
)

# ================================================================================================
# END VERBATIM EXTRACT (select the frozen 80-image holdout + save manifest/quotas)
# ================================================================================================


# --------------------------------------------------------------------------------------------------
# Additional explicit integrity assertions required by this paper notebook's spec.
# These re-state (in one place, and fail loudly) properties already implied by the
# verbatim asserts above, plus a few new ones (all path-existence / provenance, no new logic).
# --------------------------------------------------------------------------------------------------

assert len(holdout) == 80, f"Expected 80 holdout images, found {len(holdout)}"
assert (holdout["holdout_group"] == "A0_FLAGGED").sum() == 40, "Expected 40 A0_FLAGGED images"
assert (holdout["holdout_group"] == "A0_OK").sum() == 40, "Expected 40 A0_OK (control) images"
assert not set(holdout["image_id"]).intersection(development_ids), (
    "Holdout overlaps with the 84-image development set -- STOP."
)
assert set(holdout["split"].unique()).issubset({"train", "valid"}), (
    "Holdout must be Train/Validation-only provenance -- STOP."
)
assert "test" not in set(holdout["split"].unique()), (
    "Locked Test image detected in holdout -- STOP. Locked Test must never be accessed."
)

_missing_paths = []
for _, _row in holdout.iterrows():
    _p = DATASET_ROOT / _row["relative_path"]
    if not _p.exists():
        _missing_paths.append(str(_p))
assert not _missing_paths, f"Unresolved holdout image paths: {_missing_paths}"

print()
print("=" * 100)
print("HOLDOUT INTEGRITY CHECKS -- ALL PASSED")
print("=" * 100)
print("N total            :", len(holdout))
print("N A0_FLAGGED        :", (holdout["holdout_group"] == "A0_FLAGGED").sum())
print("N A0_OK (control)   :", (holdout["holdout_group"] == "A0_OK").sum())
print("Overlap w/ 84-dev set:", len(set(holdout["image_id"]).intersection(development_ids)))
print("Splits present      :", sorted(holdout["split"].unique().tolist()))
print("All image paths resolve: True (checked above)")
print("No substitutions: holdout is regenerated by the exact verbatim sampling function, not hand-edited.")


In [ ]:
# ==================================================================================================
# CELL 6 -- FREEZE MANIFEST
#
# Records everything needed to prove this exact run used the frozen A0/V2 implementation,
# the frozen holdout, and no undisclosed modification.
# ==================================================================================================

import inspect

FREEZE_MANIFEST = {}

FREEZE_MANIFEST["timestamp_utc"] = datetime.now(timezone.utc).isoformat()
FREEZE_MANIFEST["notebook_source_filename"] = NOTEBOOK_SOURCE_FILENAME
FREEZE_MANIFEST["archive_source_filename"] = ARCHIVE_SOURCE_FILENAME
FREEZE_MANIFEST["doar_repo_git_commit"] = commit

# --------------------------------------------------------------------------------------------
# Implementation hashes: hash the ACTUAL running source of each frozen function via
# inspect.getsource(), so the hash reflects exactly what executed in this kernel --
# not a copy that could have silently drifted from CELL 3.
# --------------------------------------------------------------------------------------------

def _sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

FREEZE_MANIFEST["implementation_hashes"] = {
    "build_a0_mask": _sha256_text(inspect.getsource(build_a0_mask)),
    "build_v2_mask": _sha256_text(inspect.getsource(build_v2_mask)),
    "estimate_support_v2": _sha256_text(inspect.getsource(estimate_support_v2)),
    "suppress_smooth_border_floods": _sha256_text(inspect.getsource(suppress_smooth_border_floods)),
    "v2_qc_status": _sha256_text(inspect.getsource(v2_qc_status)),
    "border_foreground_fraction": _sha256_text(inspect.getsource(border_foreground_fraction)),
    "cleanup_components": _sha256_text(inspect.getsource(cleanup_components)),
    "get_border_pixels": _sha256_text(inspect.getsource(get_border_pixels)),
    "robust_mad": _sha256_text(inspect.getsource(robust_mad)),
    "build_border_selector": _sha256_text(inspect.getsource(build_border_selector)),
    "border_samples_with_side_ids": _sha256_text(inspect.getsource(border_samples_with_side_ids)),
    "measurement_policy": _sha256_text(inspect.getsource(measurement_policy)),
    "proportional_stratified_sample": _sha256_text(inspect.getsource(proportional_stratified_sample)),
    "deterministic_hash": _sha256_text(inspect.getsource(deterministic_hash)),
}

FREEZE_MANIFEST["frozen_E2A0_source_sha256"] = source_sha
FREEZE_MANIFEST["expected_E2A0_source_sha256"] = EXPECTED_E2A0_SHA256

FREEZE_MANIFEST["constants"] = {
    "EXPECTED_E2A0_SHA256": EXPECTED_E2A0_SHA256,
    "development_set_size": len(development_ids),
    "holdout_total": len(holdout),
    "holdout_A0_FLAGGED": int((holdout["holdout_group"] == "A0_FLAGGED").sum()),
    "holdout_A0_OK": int((holdout["holdout_group"] == "A0_OK").sum()),
    "holdout_flagged_salt": "DOAR_E2A0_R1_V2_HOLDOUT_FLAGGED_V1",
    "holdout_ok_salt": "DOAR_E2A0_R1_V2_HOLDOUT_OK_V1",
    "STRATA": STRATA,
}

# --------------------------------------------------------------------------------------------
# Holdout manifest hash: hash the exact 80-row (image_id, split, true_class, holdout_group)
# table so any future re-run can be checked against this exact composition.
# --------------------------------------------------------------------------------------------

_holdout_manifest_text = (
    holdout[["image_id", "split", "true_class", "holdout_group"]]
    .sort_values(["holdout_group", "split", "true_class", "image_id"])
    .to_csv(index=False)
)
FREEZE_MANIFEST["holdout_manifest_sha256"] = _sha256_text(_holdout_manifest_text)

FREEZE_MANIFEST["library_versions"] = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "opencv": cv2.__version__,
}

FREEZE_MANIFEST_FILE = HOLDOUT_ROOT / "PAPER_FREEZE_MANIFEST.json"
FREEZE_MANIFEST_FILE.write_text(
    json.dumps(FREEZE_MANIFEST, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("=" * 100)
print("FREEZE MANIFEST")
print("=" * 100)
print(json.dumps(FREEZE_MANIFEST, indent=2, ensure_ascii=False))
print()
print("Saved to:", FREEZE_MANIFEST_FILE)


In [ ]:
# ==================================================================================================
# CELL 7 -- EXECUTE FROZEN A0 AND FROZEN V2 ON THE 80-IMAGE HOLDOUT
#
# This is the exact execution loop from archive cell 24, section '10. RUN FROZEN V2'.
# It calls build_a0_mask / build_v2_mask / border_foreground_fraction / v2_qc_status /
# measurement_policy -- all extracted verbatim in CELL 3 above. No simplified or
# alternate execution path is used.
# ==================================================================================================
# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via exact line-range slicing of the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: RUN FROZEN V2 -- section 10 execution loop
# ================================================================================================
result_rows = []


for _, row in tqdm(
    holdout.iterrows(),
    total=len(holdout),
    desc="Frozen V2 unseen holdout",
):

    image_path = (
        DATASET_ROOT
        /
        row[
            "relative_path"
        ]
    )


    with Image.open(
        image_path
    ) as image_file:

        image_rgb = np.asarray(
            image_file.convert(
                "RGB"
            )
        )


    a0 = build_a0_mask(
        image_rgb
    )


    v2 = build_v2_mask(
        image_rgb
    )


    a0_ratio = float(
        a0[
            "mask"
        ].mean()
    )


    v2_ratio = float(
        v2[
            "mask"
        ].mean()
    )


    border_ratio = (
        border_foreground_fraction(
            v2[
                "mask"
            ]
        )
    )


    qc_status = (
        v2_qc_status(
            foreground_ratio=
                v2_ratio,

            border_fg_ratio=
                border_ratio,

            support_regime=
                v2[
                    "support_regime"
                ],
        )
    )


    policy = measurement_policy(
        qc_status=
            qc_status,

        support_confidence=
            v2[
                "support_confidence"
            ],
    )


    result_rows.append(
        {
            "holdout_group":
                row[
                    "holdout_group"
                ],

            "image_id":
                str(
                    row[
                        "image_id"
                    ]
                ),

            "split":
                row[
                    "split"
                ],

            "true_class":
                row[
                    "true_class"
                ],

            "relative_path":
                row[
                    "relative_path"
                ],

            "a0_qc_status":
                row[
                    "qc_flag"
                ],

            "a0_foreground_fraction_saved":
                float(
                    row[
                        "foreground_fraction"
                    ]
                ),

            "a0_foreground_fraction_recreated":
                a0_ratio,

            "a0_threshold":
                float(
                    a0[
                        "threshold"
                    ]
                ),

            "v2_foreground_fraction":
                v2_ratio,

            "v2_border_foreground_fraction":
                border_ratio,

            "v2_qc_status":
                qc_status,

            "v2_support_regime":
                v2[
                    "support_regime"
                ],

            "v2_support_confidence":
                v2[
                    "support_confidence"
                ],

            "v2_measurement_policy":
                policy,

            "v2_support_radius":
                float(
                    v2[
                        "support_radius"
                    ]
                ),

            "v2_dominant_border_ratio":
                float(
                    v2[
                        "dominant_border_ratio"
                    ]
                ),

            "v2_covered_sides":
                int(
                    v2[
                        "covered_sides"
                    ]
                ),

            "v2_whole_support_fraction":
                float(
                    v2[
                        "whole_support_fraction"
                    ]
                ),

            "v2_cluster_dispersion":
                float(
                    v2[
                        "cluster_dispersion"
                    ]
                ),

            "v2_global_threshold":
                v2[
                    "global_threshold"
                ],

            "v2_local_threshold":
                float(
                    v2[
                        "local_threshold"
                    ]
                ),

            "v2_mask_strategy":
                v2[
                    "mask_strategy"
                ],
        }
    )

# ================================================================================================
# END VERBATIM EXTRACT (RUN FROZEN V2 -- section 10 execution loop)
# ================================================================================================


In [ ]:
# ==================================================================================================
# CELL 8 -- SAVE PER-IMAGE RESULTS + QC / SUPPORT / MEASUREMENT-POLICY SUMMARY TABLES
#
# Verbatim from archive cell 24, sections '10 (save)', '11', '12', '13'. Output file naming
# (E2A0_R1_V2_UNSEEN_HOLDOUT_RESULTS.csv etc.) matches the archive exactly.
# ==================================================================================================
# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via exact line-range slicing of the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: results dataframe + save to CSV
# ================================================================================================
results = pd.DataFrame(
    result_rows
)


RESULT_FILE = (
    TABLE_DIR
    / "E2A0_R1_V2_UNSEEN_HOLDOUT_RESULTS.csv"
)


results.to_csv(
    RESULT_FILE,
    index=False,
)


print()
print("✓ Frozen V2 evaluated on all 80 unseen images.")

# ================================================================================================
# END VERBATIM EXTRACT (results dataframe + save to CSV)
# ================================================================================================


# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via exact line-range slicing of the original cell source text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: QC summary + support summary + measurement-policy summary
# ================================================================================================
qc_summary = (
    results
    .groupby(
        [
            "holdout_group",
            "v2_qc_status",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="count"
    )
)


totals = (
    results
    .groupby(
        "holdout_group"
    )
    .size()
    .rename(
        "group_total"
    )
    .reset_index()
)


qc_summary = qc_summary.merge(
    totals,
    on="holdout_group",
    how="left",
)


qc_summary[
    "percentage"
] = (
    100.0
    *
    qc_summary[
        "count"
    ]
    /
    qc_summary[
        "group_total"
    ]
)


QC_SUMMARY_FILE = (
    TABLE_DIR
    / "E2A0_R1_V2_HOLDOUT_QC_SUMMARY.csv"
)


qc_summary.to_csv(
    QC_SUMMARY_FILE,
    index=False,
)


# ==================================================================================================
# 12. SUPPORT SUMMARY
# ==================================================================================================

support_summary = (
    results
    .groupby(
        [
            "holdout_group",
            "v2_support_regime",
            "v2_support_confidence",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="count"
    )
)


SUPPORT_SUMMARY_FILE = (
    TABLE_DIR
    / "E2A0_R1_V2_HOLDOUT_SUPPORT_SUMMARY.csv"
)


support_summary.to_csv(
    SUPPORT_SUMMARY_FILE,
    index=False,
)


# ==================================================================================================
# 13. MEASUREMENT POLICY SUMMARY
# ==================================================================================================

policy_summary = (
    results
    .groupby(
        [
            "holdout_group",
            "v2_measurement_policy",
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
)


policy_summary = policy_summary.merge(
    totals,
    on="holdout_group",
    how="left",
)


policy_summary[
    "percentage"
] = (
    100.0
    *
    policy_summary[
        "count"
    ]
    /
    policy_summary[
        "group_total"
    ]
)


POLICY_SUMMARY_FILE = (
    TABLE_DIR
    / "E2A0_R1_V2_HOLDOUT_MEASUREMENT_POLICY_SUMMARY.csv"
)


policy_summary.to_csv(
    POLICY_SUMMARY_FILE,
    index=False,
)


print()
print("=" * 112)
print("QC SUMMARY")
print("=" * 112)

print(
    qc_summary.to_string(
        index=False
    )
)


print()
print("=" * 112)
print("SUPPORT SUMMARY")
print("=" * 112)

print(
    support_summary.to_string(
        index=False
    )
)


print()
print("=" * 112)
print("MEASUREMENT RELIABILITY SUMMARY")
print("=" * 112)

print(
    policy_summary.to_string(
        index=False
    )
)

# ================================================================================================
# END VERBATIM EXTRACT (QC summary + support summary + measurement-policy summary)
# ================================================================================================


In [ ]:
# ==================================================================================================
# CELL 9 -- INTERNAL VISUAL QC CONTACT SHEET: ORIGINAL | A0 MASK | FROZEN V2 MASK
#
# Uses create_holdout_contact_sheets (extracted verbatim in CELL 3) exactly as the archive
# invokes it -- one PDF/PNG set for the 40 A0_FLAGGED images, one for the 40 A0_OK images,
# 4 cases per page, three columns per case: ORIGINAL | A0 | FROZEN V2.
# ==================================================================================================
# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via exact line-range slicing / ast extraction of the original cell source
# text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: flagged_results / ok_results split + contact-sheet generation calls
# ================================================================================================
flagged_results = (
    results[
        results[
            "holdout_group"
        ]
        == "A0_FLAGGED"
    ]
    .copy()
)


ok_results = (
    results[
        results[
            "holdout_group"
        ]
        == "A0_OK"
    ]
    .copy()
)


flagged_contact = (
    create_holdout_contact_sheets(
        flagged_results,
        FLAGGED_CONTACT_DIR,
        "E2A0_R1_V2_UNSEEN_FLAGGED",
    )
)


control_contact = (
    create_holdout_contact_sheets(
        ok_results,
        CONTROL_CONTACT_DIR,
        "E2A0_R1_V2_UNSEEN_CONTROLS",
    )
)

# ================================================================================================
# END VERBATIM EXTRACT (flagged_results / ok_results split + contact-sheet generation calls)
# ================================================================================================


print()
print("Flagged-group contact sheet PDF:", flagged_contact["pdf"])
print("OK-group (control) contact sheet PDF:", control_contact["pdf"])

if flagged_contact["png_pages"]:
    display(IPImage(filename=str(flagged_contact["png_pages"][0])))
if control_contact["png_pages"]:
    display(IPImage(filename=str(control_contact["png_pages"][0])))


In [ ]:
# ==================================================================================================
# CELL 10 -- BLANK HUMAN VISUAL-REVIEW TEMPLATE (FROZEN RUBRIC)
#
# The rubric below (GOOD / PARTIAL / FAILED / OVERSEGMENTED / UNDERSEGMENTED /
# APPROPRIATELY_AMBIGUOUS / UNCERTAIN) is copied verbatim from archive cell 24's own
# comment block ('14. MANUAL VISUAL REVIEW TABLE'). This IS the frozen rubric found in the
# archive -- it is a fixed vocabulary of allowed labels, not a scored/weighted questionnaire.
# No additional review criteria have been invented beyond what is documented here.
#
# This cell only WRITES the blank template (all review columns empty / review_complete=False).
# A human reviewer must open E2A0_R1_V2_HOLDOUT_VISUAL_REVIEW.csv, fill in
# visual_mask_quality / support_classification_quality / visual_notes for all 80 rows using
# only the allowed vocabulary above, and set review_complete=True before CELL 11 can run.
# ==================================================================================================
# ================================================================================================
# PROVENANCE NOTE (added for this frozen paper notebook -- NOT part of the archived source)
# The code below this banner, up to the matching CLOSING banner, is copied VERBATIM
# (byte-for-byte, via exact line-range slicing / ast extraction of the original cell source
# text) from:
#   colab/archive/DOAR_21_8_26_V1.ipynb
#   cell index 24  (E2-A0-R1-V2 FROZEN UNSEEN QC HOLDOUT)
#   item: manual visual review rubric comment + blank review template
# ================================================================================================
# ==================================================================================================
# 14. MANUAL VISUAL REVIEW TABLE
#
# DO NOT AUTO-FILL THESE.
#
# Allowed mask quality:
#
# GOOD
# PARTIAL
# FAILED
# OVERSEGMENTED
# UNDERSEGMENTED
# APPROPRIATELY_AMBIGUOUS
# UNCERTAIN
#
# The final decision comes from this visual review.
# ==================================================================================================

review = results.copy()


review[
    "visual_mask_quality"
] = ""


review[
    "support_classification_quality"
] = ""


review[
    "visual_notes"
] = ""


review[
    "review_complete"
] = False


REVIEW_FILE = (
    TABLE_DIR
    / "E2A0_R1_V2_HOLDOUT_VISUAL_REVIEW.csv"
)


review.to_csv(
    REVIEW_FILE,
    index=False,
)

# ================================================================================================
# END VERBATIM EXTRACT (manual visual review rubric comment + blank review template)
# ================================================================================================


print()
print("Blank visual-review template written to:", REVIEW_FILE)
print("Allowed visual_mask_quality / support_classification_quality values:")
print("  GOOD, PARTIAL, FAILED, OVERSEGMENTED, UNDERSEGMENTED, APPROPRIATELY_AMBIGUOUS, UNCERTAIN")
print("Fill in all 80 rows and set review_complete=True before running CELL 11.")


In [ ]:
# ==================================================================================================
# CELL 11 -- VISUAL-REVIEW-DEPENDENT STATISTICS (CONDITIONAL / MANUAL GATE)
#
# This cell only computes anything if a completed review file exists AND every row has
# review_complete == True. It is new glue code written for this paper notebook (the archive
# itself does not compute these summary statistics anywhere for the frozen holdout -- see
# CELL 12's honesty note on why no archive-documented numeric rule is being applied here).
# The metrics requested (recovery/40, retention/40, oversegmentation rate, review-required
# rate, unreliable/failed rate, Wilson 95% CI) are standard proportion statistics computed
# from the archive's OWN rubric labels (GOOD/PARTIAL/FAILED/OVERSEGMENTED/UNDERSEGMENTED/
# APPROPRIATELY_AMBIGUOUS/UNCERTAIN) -- consistent with the "recovered"/"retained" definitions
# archive cell 22 used for the R1-V1 pilot gate (GOOD or PARTIAL = recovered/retained).
# ==================================================================================================

from math import sqrt

def wilson_ci(successes, n, z=1.959963984540054):
    """95% Wilson score interval for a binomial proportion."""
    if n == 0:
        return (float("nan"), float("nan"))
    phat = successes / n
    denom = 1 + z ** 2 / n
    center = (phat + z ** 2 / (2 * n)) / denom
    half_width = (
        z * sqrt((phat * (1 - phat) + z ** 2 / (4 * n)) / n)
    ) / denom
    return (max(0.0, center - half_width), min(1.0, center + half_width))


if not REVIEW_FILE.exists():
    print("Human visual review file not found:", REVIEW_FILE)
    print("SKIPPING CELL 11 -- run CELL 10 first, then complete the human review.")
else:
    review_done = pd.read_csv(REVIEW_FILE)
    review_done["image_id"] = review_done["image_id"].astype(str)

    if not bool(review_done["review_complete"].all()):
        n_incomplete = int((~review_done["review_complete"].astype(bool)).sum())
        print(f"Visual review is INCOMPLETE ({n_incomplete}/{len(review_done)} rows not marked review_complete=True).")
        print("SKIPPING CELL 11 -- finish the human review before computing statistics.")
    else:
        RECOVERED_LABELS = {"GOOD", "PARTIAL"}
        RETAINED_LABELS = {"GOOD", "PARTIAL"}

        flagged_review = review_done[review_done["holdout_group"] == "A0_FLAGGED"].copy()
        ok_review = review_done[review_done["holdout_group"] == "A0_OK"].copy()

        assert len(flagged_review) == 40, f"Expected 40 A0_FLAGGED review rows, found {len(flagged_review)}"
        assert len(ok_review) == 40, f"Expected 40 A0_OK review rows, found {len(ok_review)}"

        n_recovered = int(flagged_review["visual_mask_quality"].isin(RECOVERED_LABELS).sum())
        n_retained = int(ok_review["visual_mask_quality"].isin(RETAINED_LABELS).sum())
        n_oversegmented = int((review_done["visual_mask_quality"] == "OVERSEGMENTED").sum())
        n_review_required = int((review_done["v2_measurement_policy"] == "MASK_REVIEW_REQUIRED").sum())
        n_unreliable_or_failed = int(
            (review_done["v2_measurement_policy"] == "MASK_UNRELIABLE").sum()
            + (review_done["visual_mask_quality"] == "FAILED").sum()
        )

        recovery_rate = n_recovered / 40
        retention_rate = n_retained / 40
        oversegmentation_rate = n_oversegmented / len(review_done)
        review_required_rate = n_review_required / len(review_done)
        unreliable_failed_rate = n_unreliable_or_failed / len(review_done)

        recovery_ci = wilson_ci(n_recovered, 40)
        retention_ci = wilson_ci(n_retained, 40)
        oversegmentation_ci = wilson_ci(n_oversegmented, len(review_done))
        review_required_ci = wilson_ci(n_review_required, len(review_done))
        unreliable_failed_ci = wilson_ci(n_unreliable_or_failed, len(review_done))

        VISUAL_REVIEW_STATS = {
            "flagged_case_recovery": {"n": n_recovered, "of": 40, "rate": recovery_rate, "wilson_95ci": recovery_ci},
            "control_retention": {"n": n_retained, "of": 40, "rate": retention_rate, "wilson_95ci": retention_ci},
            "oversegmentation": {"n": n_oversegmented, "of": len(review_done), "rate": oversegmentation_rate, "wilson_95ci": oversegmentation_ci},
            "review_required": {"n": n_review_required, "of": len(review_done), "rate": review_required_rate, "wilson_95ci": review_required_ci},
            "unreliable_or_failed": {"n": n_unreliable_or_failed, "of": len(review_done), "rate": unreliable_failed_rate, "wilson_95ci": unreliable_failed_ci},
        }

        VISUAL_REVIEW_STATS_FILE = HOLDOUT_ROOT / "PAPER_VISUAL_REVIEW_STATS.json"
        VISUAL_REVIEW_STATS_FILE.write_text(
            json.dumps(VISUAL_REVIEW_STATS, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

        print("=" * 100)
        print("VISUAL-REVIEW STATISTICS")
        print("=" * 100)
        print(json.dumps(VISUAL_REVIEW_STATS, indent=2, ensure_ascii=False))
        print()
        print("Saved to:", VISUAL_REVIEW_STATS_FILE)


## CELL 12 — Numerical acceptance rule for the frozen unseen holdout

**Finding: the archive documents NO numerical acceptance rule for the E2-A0-R1-V2 FROZEN
UNSEEN QC HOLDOUT experiment itself (archive cell index 24).** This section states that
honestly and quotes everything the archive actually says, rather than inventing a threshold.

### What archive cell 24 (the frozen holdout) actually says

Its `protocol["decision_rule"]` field (written verbatim to
`E2A0_R1_V2_UNSEEN_HOLDOUT_PROTOCOL.json` by the archive) reads:

> "If frozen V2 shows no catastrophic systematic failure on the unseen holdout, accept it
> as the confidence-aware deterministic baseline. Document residual limitations rather than
> retuning after seeing the holdout."

This is a **qualitative** rule ("no catastrophic systematic failure") with no numeric
threshold attached — no minimum recovery percentage, no maximum failure rate, nothing that
could be mechanically applied to the CELL 11 statistics. Consequently **CELL 12 does not
compute a pass/fail decision** — doing so would require inventing a number the archive does
not contain, which this task explicitly forbids.

### What numeric criteria DO exist elsewhere in the archive (and why they do not apply here)

Two earlier, **development-stage** pilots (both on the 84-image *development* set, not the
80-image *unseen holdout*) did declare numeric gates:

**1. Archive cell 22 (`E2-A0-R1-PILOT`, i.e. the R1-V1 pilot) — `acceptance_gate`:**

```python
acceptance_gate = {
    "minimum_failure_recovery_rate":
        0.85,

    "minimum_OK_control_retention_rate":
        0.90,

    "maximum_control_oversegmentation_rate":
        0.05,

    "recovered_visual_scores": [
        "GOOD",
        "PARTIAL",
    ],

    "retained_visual_scores": [
        "GOOD",
        "PARTIAL",
    ],

    "important_requirement":
        (
            "No obvious systematic failure by class, "
            "drawing medium, background or acquisition type."
        ),

    "decision_is_automatic":
        False,
}
```

This gate is explicitly scoped to R1 (the first, V1, correction) and explicitly labelled
`"decision_is_automatic": False`. R1-V1 was itself superseded during development after
visual review showed it caused background flooding on photographed pages (see archive cell
23's own header comment: *"R1-V1 sometimes flooded backgrounds / shadows / photographed
paper"*) — which is precisely why V2 was developed. Applying V1's numeric gate to V2's
holdout results would be applying a rule to an experiment it was never written for.

**2. Archive cell 23 (`E2-A0-R1-V2-PILOT`) — `development_gate`:**

```python
development_gate = {

    "automatic_pass":
        False,

    "requirements": [

        (
            "Recover known ordinary paper-based drawings "
            "that A0 incorrectly missed."
        ),

        (
            "Reduce V1 background flooding on photographed "
            "paper, shadows and gradients."
        ),

        (
            "Avoid assuming all supports are bright."
        ),

        (
            "Correctly flag uncertain/full-canvas cases "
            "as AMBIGUOUS_SUPPORT when support cannot be "
            "reliably identified."
        ),

        (
            "Avoid substantial regressions on existing "
            "reasonable control masks."
        ),
    ],

    "next_if_visual_development_passes":
        (
            "Freeze V2 before creating unseen "
            "80-image QC holdout."
        ),

    "next_if_visual_development_fails":
        (
            "Do not expose unseen holdout. "
            "Diagnose V2 development failures first."
        ),
}
```

Note `"automatic_pass": False` and the total absence of any percentage/threshold — V2's own
development gate is purely qualitative, by design (`next_if_visual_development_passes`:
*"Freeze V2 before creating unseen 80-image QC holdout"*). This confirms that even at the
development stage, V2 was never assigned a numeric bar; only R1-V1 (cell 22) was, and that
gate was abandoned along with V1.

### Conclusion for this notebook

No numeric acceptance rule from the archive is applicable to the CELL 11 statistics computed
on the 80-image frozen holdout. CELL 11's numbers (recovery/40, retention/40, oversegmentation
rate, review-required rate, unreliable/failed rate, and their Wilson 95% CIs) are reported for
the paper as descriptive results, evaluated against the archive's own qualitative standard —
"no catastrophic systematic failure" — by human judgment during the visual review in CELL 10,
not by a formula in this notebook.


In [ ]:
# ==================================================================================================
# CELL 13 -- DETERMINISTIC SELECTION OF THE TWO PUBLICATION EXAMPLE IMAGES
#
# New glue code (the archive does not itself select "the two paper example images" --
# that is specific to this paper notebook), but the SORT KEY and SELECTION RULE are fully
# explicit, deterministic, and documented here -- no random sampling.
#
# Sort key: image_id, ascending, lexicographic string sort (str.sort / pandas default for
# object dtype). This is the same image_id field used throughout the archive's own
# deterministic-ordering logic (e.g. deterministic_hash(image_id, salt) in CELL 3).
#
# Case A = lowest-sorted image_id, among A0_FLAGGED holdout images, whose visual_mask_quality
#          is in {"GOOD", "PARTIAL"} (the archive's own "recovered" definition -- see CELL 12).
# Case B = lowest-sorted image_id, among all 80 holdout images, whose v2_measurement_policy is
#          "MASK_REVIEW_REQUIRED" (a "difficult / review-required case").
#          If no such case exists, fall back to: lowest-sorted image_id among A0_OK holdout
#          images whose visual_mask_quality is in {"GOOD", "PARTIAL"} (a "retained control").
# ==================================================================================================

if not REVIEW_FILE.exists():
    print("Human visual review file not found:", REVIEW_FILE)
    print("SKIPPING CELL 13 -- run CELL 10 first, then complete the human review.")
else:
    _review = pd.read_csv(REVIEW_FILE)
    _review["image_id"] = _review["image_id"].astype(str)

    if not bool(_review["review_complete"].all()):
        print("Visual review is INCOMPLETE -- SKIPPING CELL 13.")
    else:
        RECOVERED_OR_RETAINED_LABELS = {"GOOD", "PARTIAL"}

        _recovered_flagged = (
            _review[
                (_review["holdout_group"] == "A0_FLAGGED")
                & (_review["visual_mask_quality"].isin(RECOVERED_OR_RETAINED_LABELS))
            ]
            .sort_values("image_id")
            .reset_index(drop=True)
        )

        _review_required = (
            _review[_review["v2_measurement_policy"] == "MASK_REVIEW_REQUIRED"]
            .sort_values("image_id")
            .reset_index(drop=True)
        )

        _retained_control = (
            _review[
                (_review["holdout_group"] == "A0_OK")
                & (_review["visual_mask_quality"].isin(RECOVERED_OR_RETAINED_LABELS))
            ]
            .sort_values("image_id")
            .reset_index(drop=True)
        )

        case_a = _recovered_flagged.iloc[0] if len(_recovered_flagged) else None
        if len(_review_required):
            case_b = _review_required.iloc[0]
            case_b_source = "lowest sorted MASK_REVIEW_REQUIRED case (any holdout group)"
        elif len(_retained_control):
            case_b = _retained_control.iloc[0]
            case_b_source = "no review-required case existed; fell back to lowest sorted retained A0_OK control"
        else:
            case_b = None
            case_b_source = "no qualifying case found for Case B"

        PUBLICATION_EXAMPLES = {
            "sort_key": "image_id (ascending, lexicographic)",
            "case_a": {
                "role": "lowest sorted qualifying recovered A0_FLAGGED case",
                "image_id": None if case_a is None else str(case_a["image_id"]),
                "relative_path": None if case_a is None else str(case_a["relative_path"]),
            },
            "case_b": {
                "role": case_b_source,
                "image_id": None if case_b is None else str(case_b["image_id"]),
                "relative_path": None if case_b is None else str(case_b["relative_path"]),
            },
        }

        PUBLICATION_EXAMPLES_FILE = HOLDOUT_ROOT / "PAPER_PUBLICATION_EXAMPLES.json"
        PUBLICATION_EXAMPLES_FILE.write_text(
            json.dumps(PUBLICATION_EXAMPLES, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

        print("=" * 100)
        print("DETERMINISTIC PUBLICATION EXAMPLE SELECTION")
        print("=" * 100)
        print(json.dumps(PUBLICATION_EXAMPLES, indent=2, ensure_ascii=False))
        print()
        print("Saved to:", PUBLICATION_EXAMPLES_FILE)


In [ ]:
# ==================================================================================================
# CELL 14 -- PAPER-READY OUTPUT GENERATION + paper_results_summary.md
#
# New glue code for this paper notebook: assembles everything computed in CELLS 4-13 into a
# single markdown summary for the thesis/paper. Every number it writes was computed earlier
# in this notebook (or is explicitly reported as "not available" if the human review step was
# skipped) -- nothing here recomputes or re-derives a different value.
# ==================================================================================================

PAPER_SUMMARY_FILE = HOLDOUT_ROOT / "paper_results_summary.md"

_lines = []
_lines.append("# E2-A0-R1-V2 Frozen Unseen QC Holdout -- Paper Results Summary")
_lines.append("")
_lines.append(f"Generated: {datetime.now(timezone.utc).isoformat()}")
_lines.append("")
_lines.append(f"Notebook: {NOTEBOOK_SOURCE_FILENAME}")
_lines.append(f"Archive source: {ARCHIVE_SOURCE_FILENAME}")
_lines.append(f"DOAR repo git commit: {commit if commit else 'UNRESOLVABLE at runtime'}")
_lines.append("")
_lines.append("## Frozen artifact identity")
_lines.append("")
_lines.append(f"- Frozen E2-A0 feature table SHA256: `{source_sha}`")
_lines.append(f"- Expected (archive-declared) SHA256: `{EXPECTED_E2A0_SHA256}`")
_lines.append(f"- 84-image development set size: {len(development_ids)}")
_lines.append(f"- Holdout total: {len(holdout)} (40 A0_FLAGGED + 40 A0_OK)")
_lines.append(f"- Holdout / development-set overlap: {len(set(holdout['image_id']).intersection(development_ids))} (must be 0)")
_lines.append("")
_lines.append("## QC / measurement-policy summary (all 80 holdout images)")
_lines.append("")
_lines.append(qc_summary.to_markdown(index=False))
_lines.append("")
_lines.append(policy_summary.to_markdown(index=False))
_lines.append("")

if "VISUAL_REVIEW_STATS" in globals():
    _lines.append("## Visual-review statistics (human review complete)")
    _lines.append("")
    for _key, _val in VISUAL_REVIEW_STATS.items():
        _ci = _val["wilson_95ci"]
        _lines.append(
            f"- **{_key}**: {_val['n']}/{_val['of']} = {_val['rate']:.3f} "
            f"(Wilson 95% CI: [{_ci[0]:.3f}, {_ci[1]:.3f}])"
        )
    _lines.append("")
else:
    _lines.append("## Visual-review statistics")
    _lines.append("")
    _lines.append(
        "NOT AVAILABLE -- the human visual review (CELL 10) had not been completed at the "
        "time this notebook was run, so CELL 11 did not compute these statistics."
    )
    _lines.append("")

_lines.append("## Numerical acceptance rule")
_lines.append("")
_lines.append(
    "No numerical acceptance rule is documented in the archive for this experiment "
    "(the frozen 80-image unseen holdout). See CELL 12 for the full honesty note and the "
    "archive's own qualitative decision rule. The only numeric gates found in the archive "
    "(85% recovery / 90% retention / 5% oversegmentation) were declared for the earlier, "
    "since-superseded R1-V1 development pilot on the 84-image development set, not for this "
    "experiment, and are NOT applied here."
)
_lines.append("")

if "PUBLICATION_EXAMPLES" in globals():
    _lines.append("## Deterministically selected publication examples")
    _lines.append("")
    _lines.append(f"- Sort key: {PUBLICATION_EXAMPLES['sort_key']}")
    _lines.append(
        f"- Case A ({PUBLICATION_EXAMPLES['case_a']['role']}): "
        f"image_id={PUBLICATION_EXAMPLES['case_a']['image_id']}, "
        f"path={PUBLICATION_EXAMPLES['case_a']['relative_path']}"
    )
    _lines.append(
        f"- Case B ({PUBLICATION_EXAMPLES['case_b']['role']}): "
        f"image_id={PUBLICATION_EXAMPLES['case_b']['image_id']}, "
        f"path={PUBLICATION_EXAMPLES['case_b']['relative_path']}"
    )
    _lines.append("")
else:
    _lines.append("## Deterministically selected publication examples")
    _lines.append("")
    _lines.append("NOT AVAILABLE -- human visual review was not complete when this notebook was run.")
    _lines.append("")

_lines.append("## Output files produced by this notebook")
_lines.append("")
_lines.append(f"- Holdout manifest: `{HOLDOUT_MANIFEST_FILE}`")
_lines.append(f"- Per-image results: `{RESULT_FILE}`")
_lines.append(f"- QC summary: `{QC_SUMMARY_FILE}`")
_lines.append(f"- Support summary: `{SUPPORT_SUMMARY_FILE}`")
_lines.append(f"- Measurement-policy summary: `{POLICY_SUMMARY_FILE}`")
_lines.append(f"- Blank/completed visual review: `{REVIEW_FILE}`")
_lines.append(f"- Flagged-group contact sheet: `{flagged_contact['pdf']}`")
_lines.append(f"- Control-group contact sheet: `{control_contact['pdf']}`")
_lines.append(f"- Freeze manifest: `{FREEZE_MANIFEST_FILE}`")
if (HOLDOUT_ROOT / "PAPER_VISUAL_REVIEW_STATS.json").exists():
    _lines.append(f"- Visual-review stats: `{HOLDOUT_ROOT / 'PAPER_VISUAL_REVIEW_STATS.json'}`")
if (HOLDOUT_ROOT / "PAPER_PUBLICATION_EXAMPLES.json").exists():
    _lines.append(f"- Publication example selection: `{HOLDOUT_ROOT / 'PAPER_PUBLICATION_EXAMPLES.json'}`")
_lines.append(f"- This summary: `{PAPER_SUMMARY_FILE}`")
_lines.append("")

PAPER_SUMMARY_TEXT = "\n".join(_lines) + "\n"
PAPER_SUMMARY_FILE.write_text(PAPER_SUMMARY_TEXT, encoding="utf-8")

print("=" * 100)
print("PAPER RESULTS SUMMARY WRITTEN")
print("=" * 100)
print(PAPER_SUMMARY_TEXT)
print("Saved to:", PAPER_SUMMARY_FILE)
